In [0]:
%sql
--CREATE CATALOG ecommerce_catalog; 
CREATE SCHEMA IF NOT EXISTS ecommerce_catalog.bronze;
CREATE SCHEMA IF NOT EXISTS ecommerce_catalog.silver;
CREATE SCHEMA IF NOT EXISTS ecommerce_catalog.gold;
-- CREATE SCHEMA ecommerce_catalog.bronze; 
-- CREATE SCHEMA ecommerce_catalog.silver; 
-- CREATE SCHEMA ecommerce_catalog.gold;

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col
source_path = "abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source"

customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{source_path}/customers_bad.csv")
)

In [0]:
customers_bronze = (
    customers_df
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("source_file_name", col('_metadata.file_path'))
)

display(customers_bronze)

customer_id,customer_name,city,state,load_timestamp,source_file_name
C001,Arjun Retail,Hyderabad,Telangana,2026-08-14T09:49:05.860355Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C002,Sai Enterprises,Bengaluru,Karnataka,2026-08-14T09:49:05.860355Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C003,Lakshmi Stores,Chennai,Tamil Nadu,2026-08-14T09:49:05.860355Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C003,Lakshmi Stores,Chennai,Tamil Nadu,2026-08-14T09:49:05.860355Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C004,null,Pune,Maharashtra,2026-08-14T09:49:05.860355Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C005,Krishna Traders,null,Andhra Pradesh,2026-08-14T09:49:05.860355Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C006,Green Basket,Hyderabad,null,2026-08-14T09:49:05.860355Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
null,Unknown Shop,Mumbai,Maharashtra,2026-08-14T09:49:05.860355Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv


In [0]:
%sql
select current_timestamp

current_timestamp()
2026-08-14T09:51:22.923519Z


In [0]:
bronze_path = "/Volumes/ecommerce_catalog/bronze/bronze_files/customers"

customers_bronze.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(bronze_path)

In [0]:
display(dbutils.fs.ls(bronze_path))

path,name,size,modificationTime
dbfs:/Volumes/ecommerce_catalog/bronze/bronze_files/customers/_SUCCESS,_SUCCESS,0,1786701804000
dbfs:/Volumes/ecommerce_catalog/bronze/bronze_files/customers/_committed_5501805426325763572,_committed_5501805426325763572,123,1786701804000
dbfs:/Volumes/ecommerce_catalog/bronze/bronze_files/customers/_started_5501805426325763572,_started_5501805426325763572,0,1786701802000
dbfs:/Volumes/ecommerce_catalog/bronze/bronze_files/customers/part-00000-tid-5501805426325763572-b95e084e-82b4-4db9-b31c-ffba578247f0-11-1-c000.snappy.parquet,part-00000-tid-5501805426325763572-b95e084e-82b4-4db9-b31c-ffba578247f0-11-1-c000.snappy.parquet,3134,1786701804000


In [0]:
customers_df_read = spark.read.parquet(bronze_path)
display(customers_df_read)

customer_id,customer_name,city,state,load_timestamp,source_file_name
C001,Arjun Retail,Hyderabad,Telangana,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C002,Sai Enterprises,Bengaluru,Karnataka,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C003,Lakshmi Stores,Chennai,Tamil Nadu,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C003,Lakshmi Stores,Chennai,Tamil Nadu,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C004,null,Pune,Maharashtra,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C005,Krishna Traders,null,Andhra Pradesh,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C006,Green Basket,Hyderabad,null,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
null,Unknown Shop,Mumbai,Maharashtra,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv


**PRODUCTS**

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Source ADLS location
source_path = "abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source"

# Read products CSV and add Bronze metadata
products_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{source_path}/products_bad.csv")
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("source_file_name", col("_metadata.file_path"))
)

# Databricks managed Volume path
bronze_path = "/Volumes/ecommerce_catalog/bronze/bronze_files/products"

# Write as Parquet
products_bronze.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(bronze_path)

# Verify
display(spark.read.parquet(bronze_path))

product_id,product_name,category,price,load_timestamp,source_file_name
P001,Laptop,Electronics,50000.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P002,Wireless Mouse,Electronics,800.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P003,Keyboard,Electronics,1500.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P003,Keyboard,Electronics,1500.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P004,Office Chair,Furniture,-7000.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P005,null,Furniture,9000.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P006,Headphones,null,2500.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P007,Monitor,Electronics,null,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv


In [0]:
from pyspark.sql.functions import current_timestamp, col

# Source ADLS location
source_path = "abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source"

# Read orders CSV and add Bronze metadata
orders_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{source_path}/orders_bad.csv")
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("source_file_name", col("_metadata.file_path"))
)

# Databricks managed Volume path
bronze_path = "/Volumes/ecommerce_catalog/bronze/bronze_files/orders"

# Write as Parquet
orders_bronze.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(bronze_path)

# Verify
display(spark.read.parquet(bronze_path))

order_id,customer_id,product_id,order_date,quantity,load_timestamp,source_file_name
O1001,C001,P001,2026-08-01,1.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv
O1002,C002,P002,2026-08-01,3.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv
O1003,C003,P003,2026-08-02,2.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv
O1003,C003,P003,2026-08-02,2.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv
O1004,null,P004,2026-08-02,1.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv
O1005,C999,P001,2026-08-03,2.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv
O1006,C005,P999,2026-08-03,1.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv
O1007,C006,P006,2026-13-04,4.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv
O1008,C001,P002,not-a-date,2.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv
O1009,C002,P003,2026-08-05,0.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv
